# SSL400 EXP5: EfficientNetV2-S + BiLSTM + CLAHE
## ALL DATA INCLUDED IN ZIP - NO SHARING NEEDED
---
### Steps: Run cells 1 -> 8 in order


In [ ]:
# Cell 1 - MOUNT GOOGLE DRIVE & SETUP
from google.colab import drive
drive.mount('/content/drive')
import os, shutil

# =====================================================
PACKAGE_DIR  = '/content/drive/MyDrive/SSL400_EXP5'
DRIVE_SAVE   = '/content/drive/MyDrive/SSL400_EXP5/models/experiment_5'
# =====================================================

LOCAL = '/content/ssl400'
os.makedirs(LOCAL, exist_ok=True)

print('Copying code and ALL data to local SSD (this might take a minute)...')
if os.path.exists(LOCAL):
    shutil.rmtree(LOCAL)
shutil.copytree(PACKAGE_DIR, LOCAL)

os.makedirs(DRIVE_SAVE, exist_ok=True)
os.chdir(LOCAL)

raw_dir = os.path.join(LOCAL, 'data', 'raw')
classes = os.listdir(raw_dir) if os.path.exists(raw_dir) else []
print('\nWorking directory:', os.getcwd())
print('Raw video classes found:', classes)
total = sum(len(os.listdir(os.path.join(raw_dir, c))) 
            for c in classes if os.path.isdir(os.path.join(raw_dir, c)))
print('Total video files ready for processing:', total)
print('Setup complete!')


In [ ]:
# Cell 1.5 - HOTFIX PYTHON FILES
import os

print("Applying hotfixes to Python files...")
factory_py = "/content/ssl400/src/enhancement/enhancement_factory.py"
if os.path.exists(factory_py):
    with open(factory_py, "r") as f:
        fc = f.read()
    if "elif exp_id == 5:" not in fc:
        fc = fc.replace("elif exp_id == 4:\n        from enhancement.hybrid import enhance_hybrid\n        return enhance_hybrid",
                        "elif exp_id == 4:\n        from enhancement.hybrid import enhance_hybrid\n        return enhance_hybrid\n\n    elif exp_id == 5:\n        from enhancement.clahe_gamma import enhance_clahe_gamma\n        return enhance_clahe_gamma")
        fc = fc.replace("between 1 and 4.", "between 1 and 5.")
        with open(factory_py, "w") as f: f.write(fc)

eval_py = "/content/ssl400/src/evaluation/evaluate.py"
if os.path.exists(eval_py):
    with open(eval_py, "r") as f:
        ec = f.read()
    if "choices=[1, 2, 3, 4, 5]" not in ec:
        ec = ec.replace("choices=[1, 2, 3, 4]", "choices=[1, 2, 3, 4, 5]")
        with open(eval_py, "w") as f: f.write(ec)

vid_py = "/content/ssl400/src/data/video_to_frames.py"
if os.path.exists(vid_py):
    with open(vid_py, "r") as f:
        vc = f.read()
    if "choices=[1, 2, 3, 4, 5]" not in vc:
        vc = vc.replace("choices=[1, 2, 3, 4]", "choices=[1, 2, 3, 4, 5]")
        with open(vid_py, "w") as f: f.write(vc)
print("Hotfixes applied!")


In [ ]:
# Cell 2 - INSTALL DEPENDENCIES
!pip install tf-keras tf-models-official ultralytics --quiet
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
print('Dependencies installed!')

In [ ]:
# Cell 3 - TRAIN/VAL/TEST SPLITS (already included, just verify)
import csv
with open('/content/ssl400/data/splits/train_split.csv') as f:
    n = sum(1 for _ in csv.DictReader(f))
print('Splits loaded successfully! Train samples:', n)


In [ ]:
# Cell 4 - PROCESS RAW VIDEOS -> .NPY FRAMES
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
proc_dir  = '/content/ssl400/data/processed/exp5_clahe_gamma'
drive_npy = '/content/drive/MyDrive/SSL400_EXP5/data/processed/exp5_clahe_gamma'
if os.path.exists(drive_npy):
    print('Found backed up .npy on Drive. Copying to local...')
    import shutil
    if os.path.exists(proc_dir): shutil.rmtree(proc_dir, ignore_errors=True)
    shutil.copytree(drive_npy, proc_dir, dirs_exist_ok=True)
    npy_count = sum(len([f for f in os.listdir(os.path.join(proc_dir, c)) if f.endswith(".npy")]) for c in os.listdir(proc_dir) if os.path.isdir(os.path.join(proc_dir, c))) if os.path.exists(proc_dir) else 0
if npy_count > 400:
    print('Already processed! Skipping.')
else:
    print('Processing raw videos with CLAHE + Gamma (32 frames)... 30-60 mins.')
    !python /content/ssl400/src/data/video_to_frames.py --exp_id 5
    import shutil
    os.makedirs(drive_npy, exist_ok=True)
    for cid in os.listdir(proc_dir):
        cdir = os.path.join(proc_dir, cid)
        if not os.path.isdir(cdir): continue
        d_cdir = os.path.join(drive_npy, cid)
        os.makedirs(d_cdir, exist_ok=True)
        for fn in os.listdir(cdir):
            if fn.endswith(".npy"):
                shutil.copy2(os.path.join(cdir, fn), os.path.join(d_cdir, fn))


In [ ]:
# Cell 5 - BUILD EfficientNetV2-S + BiLSTM MODEL
import tensorflow as tf, numpy as np, yaml, os
try:
    import tf_keras as keras
except ImportError:
    keras = tf.keras
with open('/content/ssl400/config.yaml') as f:
    config = yaml.safe_load(f)
NUM_FRAMES  = config['frames']['num_frames']
IMG_H       = config['frames']['height']
IMG_W       = config['frames']['width']
NUM_CLASSES = config['dataset']['num_classes']
SEED        = config['project']['seed']
BATCH_SIZE  = 2
PROC_DIR    = '/content/ssl400/data/processed/exp5_clahe_gamma'
TRAIN_CSV   = '/content/ssl400/data/splits/train_split.csv'
VAL_CSV     = '/content/ssl400/data/splits/val_split.csv'
TEST_CSV    = '/content/ssl400/data/splits/test_split.csv'
LOCAL_MDL   = '/content/exp5_model'
LOG_PATH    = '/content/exp5_log.csv'
os.makedirs(LOCAL_MDL, exist_ok=True)
tf.random.set_seed(SEED)
np.random.seed(SEED)
print('Frames:', NUM_FRAMES, '| Classes:', NUM_CLASSES, '| Batch:', BATCH_SIZE)
backbone = tf.keras.applications.EfficientNetV2S(
    include_top=False, weights='imagenet',
    pooling='avg', input_shape=(IMG_H, IMG_W, 3)
)
backbone.trainable = False
inp = keras.Input(shape=(NUM_FRAMES, IMG_H, IMG_W, 3))
x   = keras.layers.TimeDistributed(backbone, name='efficientnetv2')(inp)
x   = keras.layers.TimeDistributed(keras.layers.BatchNormalization())(x)
x   = keras.layers.Bidirectional(keras.layers.LSTM(256), name='bilstm')(x)
x   = keras.layers.Dropout(0.4)(x)
out = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = keras.Model(inputs=inp, outputs=out)
print('Model parameters:', model.count_params())


In [ ]:
# Cell 6 - LOAD DATASET
import csv, numpy as np, tensorflow as tf
def load_ds(csv_path, batch_size):
    samples = []
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        cols = reader.fieldnames
        lc = 'label' if 'label' in cols else cols[-1]
        for row in reader:
            stem = os.path.splitext(row[cols[0]])[0]
            cid = str(int(row[lc]))
            p = os.path.join(PROC_DIR, cid, stem + ".npy")
            if os.path.exists(p): samples.append((p, int(row[lc])))
    print(' ', os.path.basename(csv_path), ':', len(samples), 'samples')
    def gen():
        for path, label in samples:
            frames = np.load(path).astype(np.float32)
            if frames.shape[0] != NUM_FRAMES:
                idx = np.linspace(0, frames.shape[0]-1, NUM_FRAMES, dtype=int)
                frames = frames[idx]
            frames = np.clip(frames * 127.5 + 127.5, 0, 255)
            oh = np.zeros(NUM_CLASSES, np.float32); oh[label] = 1.0
            yield frames, oh
    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((NUM_FRAMES, IMG_H, IMG_W, 3), tf.float32),
        tf.TensorSpec((NUM_CLASSES,), tf.float32)
    )).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, len(samples)
print('Loading dataset...')
train_ds, ntr = load_ds(TRAIN_CSV, BATCH_SIZE)
val_ds,   nvl = load_ds(VAL_CSV,   BATCH_SIZE)
test_ds,  nts = load_ds(TEST_CSV,  BATCH_SIZE)


In [ ]:
# Cell 7 - TRAIN (Phase 1 frozen backbone -> Phase 2 full fine-tune)
import shutil, os
DRIVE_SAVE = '/content/drive/MyDrive/SSL400_EXP5/models/experiment_5'
def sync(phase):
    os.makedirs(DRIVE_SAVE, exist_ok=True)
    for fn in os.listdir(LOCAL_MDL):
        shutil.copy2(os.path.join(LOCAL_MDL, fn), os.path.join(DRIVE_SAVE, fn))
    if os.path.exists(LOG_PATH):
        shutil.copy2(LOG_PATH, os.path.join(DRIVE_SAVE, 'training_log_' + phase + '.csv'))
    print('  Saved to Drive!')
# Phase 1
print('PHASE 1: Frozen EfficientNetV2-S backbone')
model.compile(optimizer=keras.optimizers.Adam(0.001, clipnorm=1.0),
              loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=[
    keras.callbacks.EarlyStopping('val_loss', patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(LOCAL_MDL + '/best_model_phase1.keras', monitor='val_loss', save_best_only=True, verbose=1),
    keras.callbacks.CSVLogger(LOG_PATH, append=False),
    keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=5, verbose=1)
])
sync('phase1')
# Phase 2
print('PHASE 2: Full fine-tuning')
for layer in model.layers: layer.trainable = True
model.compile(optimizer=keras.optimizers.Adam(0.0001, clipnorm=1.0),
              loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
              metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=[
    keras.callbacks.EarlyStopping('val_loss', patience=25, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(LOCAL_MDL + '/best_model_phase2.keras', monitor='val_loss', save_best_only=True, verbose=1),
    keras.callbacks.CSVLogger(LOG_PATH, append=False),
    keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=8, verbose=1),
    keras.callbacks.LambdaCallback(on_epoch_end=lambda e, l: sync('phase2') if (e + 1) % 5 == 0 else None)
])
sync('phase2')


In [ ]:
# Cell 8 - EVALUATE
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
import json, numpy as np, matplotlib.pyplot as plt, seaborn as sns, pandas as pd, shutil
CLASS_NAMES = ['Thank you','Hello','Good','House','Eat','Drink','Tell','Write']
print('Evaluating...')
logits_all, labels_all = [], []
for xb, yb in test_ds:
    logits_all.append(model.predict(xb, verbose=0))
    labels_all.append(yb.numpy())
logits = np.concatenate(logits_all)
y_true = np.argmax(np.concatenate(labels_all), axis=1)
y_pred = np.argmax(logits, axis=1)
top1 = accuracy_score(y_true, y_pred)
top5 = sum(1 for t, l in zip(y_true, logits) if t in np.argsort(l)[-5:]) / len(y_true)
f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
rpt_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
rpt_str  = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)
cm = confusion_matrix(y_true, y_pred)
print('Top-1 Accuracy:', round(top1 * 100, 2), '%')
print(rpt_str)
results = {'exp_id': 5, 'exp_name': 'EfficientNetV2-S + BiLSTM + CLAHE',
           'top1_accuracy': float(top1), 'top5_accuracy': float(top5),
           'macro_f1': float(f1), 'macro_precision': float(prec), 'macro_recall': float(rec),
           'classification_report': rpt_dict}
with open('/content/exp5_metrics.json', 'w') as f: json.dump(results, f, indent=2)
np.save('/content/exp5_cm.npy', cm)
shutil.copy2('/content/exp5_metrics.json', os.path.join(DRIVE_SAVE, 'exp5_metrics.json'))
shutil.copy2('/content/exp5_cm.npy',       os.path.join(DRIVE_SAVE, 'exp5_cm.npy'))
